# 00 – Start und Projektübersicht

Dieses Notebook prüft die Projektstruktur und zeigt registrierte Szenarien und vorhandene Ergebnisse. **Es berechnet nichts und schreibt nichts.**

## Zweck und Grenzen des Modells

- **Zweck:** Gewichtete Abbildung des österreichischen Gebäudesektors über 43 200 Gebäudetypen (Geometrie × Qualität × Versorgung × PV × Flexibilität × Wetter × Speichermasse). Ein Szenario ist ein **fester Zustand** in einem Jahr: eine NGF-Verteilung über alle Typen, gebildet aus Excel-Verteilungen, optional energetisch bewertet.
- **Vergleiche** (02) stellen gespeicherte Zustände nebeneinander. **Flächenströme** (03) rekonstruieren optional eine regelkonforme Zuordnung zwischen zwei gespeicherten Zuständen einschließlich ausdrücklich vorgegebenem Neubau und Abriss und leiten daraus Maßnahmenraten ab.
- **Grenzen:** Zustände werden nicht über Zielvorgaben optimiert (die zielgeführte Entwicklung ist zurückgestellt). Flüsse und Raten sind eine zulässige Zuordnung zwischen zwei Endzuständen, keine beobachteten Maßnahmen. Aus zwei Zuständen allein ist nur die Nettoänderung bekannt; Neubau und Abriss stammen immer aus ausdrücklichen Vorgaben. Energiekennwerte gibt es nur, wenn `scenarios.parquet` vorhanden ist und die Energieanalyse ausdrücklich ausgeführt wurde.

## Standardworkflow

| Schritt | Notebook | Ergebnis |
|---|---|---|
| 1 | `00_Start` | Pfade, Pakete, Register und vorhandene Ergebnisse prüfen (nur lesen) |
| 2 | `01_Szenario` | je Szenario: Zustand bilden, prüfen, optional Energie, speichern → `results/szenarien/<ID>/` |
| 3 | `02_Szenarienvergleich` | gespeicherte Zustände vergleichen, optional exportieren → `results/vergleiche/<ID>/` |
| 4 | `03_Flaechenstroeme` | je Paar A → B: vorhandene Flüsse lesen oder Rekonstruktion ausdrücklich starten, Maßnahmenraten, optional exportieren → `results/flaechenstroeme/<ID>/` |

Jeder Schritt liest nur gespeicherte Ergebnisse des vorherigen Schritts. Berechnen, Exportieren und Ersetzen sind in jedem Notebook standardmäßig ausgeschaltet (`False`).

## Nach einer Aktualisierung der Projektdateien

Notebooks, die beim Aktualisieren geöffnet waren, **ohne Speichern schließen**, neu öffnen und den Kernel neu starten. Die Startzelle lädt die Module immer frisch aus `modell/` und bricht ab, wenn Notebook- und Modellfassung nicht zusammenpassen.

## Start

- **Was:** Projektordner finden und Modell laden.
- **Eingaben:** `config/projekt.json`
- **Rechnet oder liest:** liest nur
- **Annahmen:** Notebook liegt im Ordner `notebooks/`.
- **Interpretation:** –
- **Sperrende Fehler:** Projektordner nicht gefunden.

In [1]:
# Einheitliche Startzelle: Projektordner suchen, aktuelle Modellmodule laden, Fassung prüfen, Projektpfade lesen.
import os, sys
from pathlib import Path
try:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config' / 'projekt.json').is_file())
except StopIteration:
    raise FileNotFoundError('Projektordner nicht gefunden: Das Notebook muss im Ordner notebooks/ des Projekts liegen (config/projekt.json fehlt).') from None
MODELL = ROOT / 'modell'
if str(MODELL) not in sys.path:
    sys.path.insert(0, str(MODELL))
for _name in [n for n, m in list(sys.modules.items()) if os.path.normcase(str(getattr(m, '__file__', None) or '')).startswith(os.path.normcase(str(MODELL)))]:
    del sys.modules[_name]   # nach einer Aktualisierung immer die Module aus dem Projektordner laden (kein veralteter Kernelstand)
import fassung
print('Projektordner:', ROOT)
print(fassung.pruefe_notebook('00_Start', '1.6.3-AP6'))
import gebaeudesektor as gs
from IPython.display import display
PROJEKT = gs.lade_projekt(ROOT)
import abnahme as ab

Projektordner: c:\Users\ma0026.NB-MA0026\Documents\peq-higrube\Berechnungen\gebaeudesektormodell\Gebaeudesektor_konsolidiert\Gebaeudesektor
Fassung 1.6.3-AP6 (Notebook und Modell passen zusammen)


c:\Users\ma0026.NB-MA0026\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\ma0026.NB-MA0026\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## Projektprüfung

- **Was:** Prüft, ob Konfigurationen, Eingabedateien und Python-Pakete vorhanden sind.
- **Eingaben:** `config/projekt.json`, Dateien unter `data/`
- **Rechnet oder liest:** liest nur
- **Annahmen:** `scenarios.parquet` und `pyarrow` sind nur für die Energieanalyse nötig.
- **Interpretation:** Status FEHLT bei Pflichtdateien muss vor 01 behoben werden.
- **Sperrende Fehler:** keine (nur Anzeige)

In [2]:
display(gs.pruefe_projekt(PROJEKT))

,Bereich,Eintrag,Pfad,Status,Hinweis
0,konfiguration,szenarien,config/szenarien.json,OK,
1,konfiguration,vergleiche,config/vergleiche.json,OK,
2,konfiguration,uebergaenge,config/flaechenstroeme/uebergaenge.json,OK,
3,konfiguration,rekonstruktionen,config/flaechenstroeme/rekonstruktionen.json,OK,
4,konfiguration,bestandsentwicklung,data/eingaben/bestandsentwicklung/Bestandsentw...,OK,
5,konfiguration,massnahmen,config/flaechenstroeme/massnahmen.json,OK,
6,dateien,basis,data/eingaben/annahmen.json,OK,
7,dateien,statistik,data/referenz/statistik/Statistik_Austria_2024...,OK,
8,dateien,katalog,data/referenz/typologien_43200.csv,OK,
9,dateien,parquet,data/simulation/scenarios.parquet,OK,nur für die Energieanalyse erforderlich


## Registrierte Szenarien

- **Was:** Szenarien aus `config/szenarien.json`.
- **Eingaben:** `config/szenarien.json`
- **Rechnet oder liest:** liest nur
- **Annahmen:** Szenario-ID = Ergebnisordnername.
- **Interpretation:** Nur registrierte Szenarien können in 01 berechnet werden.
- **Sperrende Fehler:** ungültiges Szenarioregister

In [3]:
display(gs.registrierte_szenarien(PROJEKT))

,Szenario_ID,Typ,Jahr,Statistikjahr,Ziel_NGF_m2,Verteilungsdatei,Datei_vorhanden,Beschreibung
0,Bestand_2024,Bestandszustand,2024,2024,NaN,data/eingaben/verteilungen/Verteilung_Bestand_...,True,Kalibrierter Gebäudebestand 2024
1,Transition_2040,Zukunftszustand,2040,2024,NaN,data/eingaben/verteilungen/Verteilung_Transiti...,True,Illustratives Transformations-Zielbild 2040 be...
2,Transition_2040_Wachstum,Zukunftszustand,2040,2024,950000000.0,data/eingaben/verteilungen/abgeleitet/Verteilu...,True,Illustratives Transformations-Zielbild 2040 mi...
3,Transition_2040_ambitioniert,Zukunftszustand,2040,2024,950000000.0,data/eingaben/verteilungen/abgeleitet/Verteilu...,False,"Flächenentwicklung wie Wachstum, hohe Sanierun..."


## Abgeleitete Prioren (AP6.3)

- **Was:** Szenarien, deren Verteilung aus einer Eingabedatei mit leeren Prior-Zellen abgeleitet wird, und Aktualität der abgeleiteten Datei.
- **Eingaben:** Profilangabe `prioren` in `config/szenarien.json`, Eingabedatei, abgeleitete Datei, gespeicherte Basis
- **Rechnet oder liest:** liest nur (vergleicht Prüfsummen und Zustands-ID der Basis; keine Ableitung)
- **Annahmen:** Szenarien ohne `prioren` erscheinen nicht.
- **Interpretation:** AKTUELL: abgeleitete Datei passt zu Eingabe, Basis, Regeln und Bestandsentwicklung. FEHLT/VERALTET: in 01 mit `PRIOREN_ABLEITEN=True` neu ableiten (Gründe lesen).
- **Sperrende Fehler:** keine (nur Anzeige)

In [4]:
import prioren as pr
prioren_status = pr.statustabelle(PROJEKT)
display(prioren_status) if len(prioren_status) else print('Kein Szenario mit abgeleiteten Prioren registriert.')

,Szenario_ID,Status,Eingabe,Abgeleitete_Datei,Basis,Gruende
0,Transition_2040_Wachstum,VERALTET,data/eingaben/verteilungen/Verteilung_Transiti...,data/eingaben/verteilungen/abgeleitet/Verteilu...,Bestand_2024,Bestandsentwicklung.xlsx geändert
1,Transition_2040_ambitioniert,FEHLT,data/eingaben/verteilungen/Verteilung_Transiti...,data/eingaben/verteilungen/abgeleitet/Verteilu...,Bestand_2024,abgeleitete Datei fehlt


## Vorhandene Ergebnisse

- **Was:** Status gespeicherter Szenarioergebnisse unter `results/szenarien/`.
- **Eingaben:** gespeicherte Ergebnisordner
- **Rechnet oder liest:** liest nur (prüft SHA-256 der Ausgaben)
- **Annahmen:** –
- **Interpretation:** LESBAR = Ergebnis kann in 02/03 verwendet werden (Spalte `Kennwerte` = True nötig für 02). FEHLERHAFT (Hinweis lesen: Altformat, fehlende oder veränderte Dateien) = in 01 neu speichern. NICHT_LESBAR = Zugriffsrechte des Ordners prüfen.
- **Sperrende Fehler:** keine (nur Anzeige)

In [5]:
display(gs.ergebnisuebersicht(PROJEKT))

,Szenario_ID,Ordner,Status,Jahr,NGF_m2,Positive_Typen,Zustand_ID,Energie,Kennwerte,Integritaet
0,Bestand_2024,results/szenarien/Bestand_2024,LESBAR,2024,889872558.0,14640,6d9c1ee883204a04,Ausgeführt,True,OK (SHA-256 der Ausgaben geprüft)
1,Transition_2040,results/szenarien/Transition_2040,LESBAR,2040,889872558.0,16320,e07bff047ea3de97,Ausgeführt,True,OK (SHA-256 der Ausgaben geprüft)
2,Transition_2040_Wachstum,results/szenarien/Transition_2040_Wachstum,LESBAR,2040,950000000.0,15336,cbf1cf4d08e5df11,Ausgeführt,True,OK (SHA-256 der Ausgaben geprüft)


## Abnahmestand (AP7)

- **Was:** Übersicht aller registrierten Szenarien, Vergleiche und Flächenstrompaare für die lokale Gesamtprüfung.
- **Eingaben:** gespeicherte Ergebnisse unter `results/`, Register unter `config/`
- **Rechnet oder liest:** liest nur (nur bei `ABNAHMESTAND_PRUEFEN=True`): prüft SHA-256, ob gespeicherte Flüsse zu den aktuellen Zuständen gehören, und wertet die Maßnahmenraten aus den gespeicherten Flüssen erneut aus; keine Optimierung, kein Schreiben
- **Annahmen:** Dauer etwa eine halbe Minute je Flächenstrompaar.
- **Interpretation:** OK: abnahmebereit. FEHLT: Schritt im jeweiligen Notebook ausführen und speichern. HINWEIS: vor der aktuellen Fassung gespeichert oder Status ≠ MACHBAR (Detail lesen). FEHLERHAFT/FEHLER: neu erstellen bzw. rekonstruieren.
- **Sperrende Fehler:** –

In [6]:
ABNAHMESTAND_PRUEFEN = False   # Abnahmestand aller gespeicherten Ergebnisse anzeigen (nur lesen)
if ABNAHMESTAND_PRUEFEN:
    display(ab.abnahmestand(PROJEKT))
else:
    print('ABNAHMESTAND_PRUEFEN=False: keine Prüfung.')

ABNAHMESTAND_PRUEFEN=False: keine Prüfung.
